In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_DIR = Path(
    r"C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading"
)

DATA_PATH = PROJECT_DIR / "ml/grading/data/processed/cleaned_grading_dataset.xlsx"
SPLIT_PATH = (
    PROJECT_DIR
    / "ml/grading/image/data/processed/splits/multimodal_shared_split.csv"
)

SCRIPTS_DIR = PROJECT_DIR / "ml/grading/scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import modeling_setup as ms

tabular_df = pd.read_excel(DATA_PATH)
shared_split = pd.read_csv(SPLIT_PATH)

# Normalize IDs consistently with the multimodal alignment
def normalize_id(value):
    if pd.isna(value):
        return None
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

tabular_df["Normalized_ID"] = tabular_df["Tissue_ID"].apply(normalize_id)

# Join the exact IDs and split assignments used by the multimodal model
tabular_df = tabular_df.merge(
    shared_split[["Normalized_ID", "Split"]],
    on="Normalized_ID",
    how="inner",
    validate="one_to_one"
)

assert tabular_df["Normalized_ID"].is_unique
assert tabular_df[ms.TARGET].notna().all()
assert len(tabular_df) == 1009

print("Dataset shape:", tabular_df.shape)
print("\nSplit sizes:")
print(tabular_df["Split"].value_counts())

print("\nGrade distribution:")
print(pd.crosstab(tabular_df["Split"], tabular_df[ms.TARGET]))

print("\nFeature count:", len(ms.ALL_FEATURES))
print("Missing target values:", tabular_df[ms.TARGET].isna().sum())

Dataset shape: (1009, 118)

Split sizes:
Split
train         706
test          152
validation    151
Name: count, dtype: int64

Grade distribution:
Tissue_Grade_Clean    A   A+    B
Split                            
test                 47   80   25
train               220  371  115
validation           47   79   25

Feature count: 24
Missing target values: 0


In [2]:
# Separate records by the shared split
train_mask = tabular_df["Split"] == "train"
val_mask = tabular_df["Split"] == "validation"
test_mask = tabular_df["Split"] == "test"

X_train = tabular_df.loc[train_mask, ms.ALL_FEATURES]
X_val = tabular_df.loc[val_mask, ms.ALL_FEATURES]
X_test = tabular_df.loc[test_mask, ms.ALL_FEATURES]

y_train = tabular_df.loc[train_mask, ms.TARGET]
y_val = tabular_df.loc[val_mask, ms.TARGET]
y_test = tabular_df.loc[test_mask, ms.TARGET]

# Fit preprocessing on training data only
tabular_preprocessor = ms.make_preprocessor()

X_train_processed = tabular_preprocessor.fit_transform(X_train)
X_val_processed = tabular_preprocessor.transform(X_val)
X_test_processed = tabular_preprocessor.transform(X_test)

print("Processed feature shapes:")
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

print("\nTarget shapes:")
print("Train:", y_train.shape)
print("Validation:", y_val.shape)
print("Test:", y_test.shape)

assert len(X_train_processed) == 706
assert len(X_val_processed) == 151
assert len(X_test_processed) == 152

print("\nTabular preprocessing completed.")

Processed feature shapes:
Train: (706, 68)
Validation: (151, 68)
Test: (152, 68)

Target shapes:
Train: (706,)
Validation: (151,)
Test: (152,)

Tabular preprocessing completed.


In [4]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np

SEED = 42

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1
    )
}

validation_results = []
fitted_models = {}

for name, classifier in models.items():
    classifier.fit(X_train_processed, y_train)

    val_pred = classifier.predict(X_val_processed)
    val_f1 = f1_score(
        y_val,
        val_pred,
        average="macro",
        zero_division=0
    )

    fitted_models[name] = classifier
    validation_results.append({
        "Model": name,
        "Validation Macro F1": val_f1
    })

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values("Validation Macro F1", ascending=False)
    .reset_index(drop=True)
)

print(validation_results_df)

best_model_name = validation_results_df.loc[0, "Model"]
best_model = fitted_models[best_model_name]

print("Selected model:", best_model_name)
print(
    "Best validation macro F1:",
    f"{validation_results_df.loc[0, 'Validation Macro F1']:.4f}"
)

                 Model  Validation Macro F1
0        Random Forest             0.836215
1  Logistic Regression             0.721703
2          Extra Trees             0.684367
Selected model: Random Forest
Best validation macro F1: 0.8362


In [5]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_test_pred = best_model.predict(X_test_processed)

print("Selected model:", best_model_name)
print("Test samples:", len(y_test))

print(f"Accuracy:          {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_test_pred):.4f}")
print(f"Macro F1:          {f1_score(y_test, y_test_pred, average='macro'):.4f}")

print("\nClassification report:")
print(classification_report(
    y_test,
    y_test_pred,
    labels=["A", "A+", "B"],
    digits=4,
    zero_division=0
))

print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(
    y_test,
    y_test_pred,
    labels=["A", "A+", "B"]
))

Selected model: Random Forest
Test samples: 152
Accuracy:          0.9013
Balanced accuracy: 0.8928
Macro F1:          0.9061

Classification report:
              precision    recall  f1-score   support

           A     0.8837    0.8085    0.8444        47
          A+     0.8837    0.9500    0.9157        80
           B     1.0000    0.9200    0.9583        25

    accuracy                         0.9013       152
   macro avg     0.9225    0.8928    0.9061       152
weighted avg     0.9028    0.9013    0.9007       152

Confusion matrix (rows = actual, columns = predicted):
[[38  9  0]
 [ 4 76  0]
 [ 1  1 23]]


# Shared-Split Baseline Comparison

## 1. Experimental Objective

The objective of this experiment was to compare three approaches for donor cornea tissue grading:

1. **Image-only:** A custom convolutional neural network (CNN) trained on specular microscopy images.
2. **Tabular-only:** A Random Forest classifier trained on clinical, endothelial morphology, and tissue-related features.
3. **Multimodal:** A neural network combining image-derived features with preprocessed tabular features.

All three experiments used the same tissue-level train, validation, and test split to support a consistent comparison.

## 2. Dataset and Split

The aligned dataset contained **1,009 labelled tissue records** with corresponding images and specialist-reviewed tissue grades.

| Split | Total | A | A+ | B |
|---|---:|---:|---:|---:|
| Training | 706 | 220 | 371 | 115 |
| Validation | 151 | 47 | 79 | 25 |
| Test | 152 | 47 | 80 | 25 |
| **Total** | **1,009** | **314** | **530** | **165** |

The split was stratified by `Tissue_Grade_Clean`, using random seed 42.

The tabular dataset initially contained 24 selected input features. The preprocessing pipeline transformed these into 68 model-ready features.

## 3. Model Configurations

### 3.1 Image-only CNN

A custom CNN was trained using grayscale specular microscopy images resized and padded to 224 × 224 pixels.

- Input: Single-channel grayscale image.
- Architecture: Three convolutional blocks with 16, 32, and 64 channels, followed by adaptive average pooling and a fully connected classifier.
- Trainable parameters: 23,715.
- Optimizer: Adam.
- Learning rate: 0.001.
- Weight decay: 0.0001.
- Batch size: 16.
- Maximum epochs: 30.
- Early-stopping patience: 5 epochs.
- Checkpoint selection: Highest validation macro F1.

The best validation macro F1 was **0.4170 at epoch 9**. Early stopping occurred at epoch 14.

### 3.2 Tabular-only Random Forest

Three classifiers were compared using the shared training and validation sets:

- Logistic Regression: validation macro F1 = 0.7217.
- Random Forest: validation macro F1 = 0.8362.
- Extra Trees: validation macro F1 = 0.6844.

The Random Forest was selected based on validation macro F1. It used 300 trees and random seed 42. The tabular preprocessing pipeline was fitted on the training data only.

### 3.3 Multimodal Fusion Model

The multimodal model combined a custom image encoder with a tabular encoder. The image branch produced a 64-dimensional representation, while the tabular branch transformed the 68 preprocessed features into a 32-dimensional representation. The concatenated representation was passed to a fully connected classification head.

The model contained 34,819 trainable parameters. Training used Adam with a learning rate of 0.001, weight decay of 0.0001, batch size 16, and early stopping based on validation macro F1.

The best validation macro F1 was **0.8066 at epoch 21**.

## 4. Test-Set Results

| Metric | Image-only CNN | Tabular-only Random Forest | Multimodal |
|---|---:|---:|---:|
| Test accuracy | 0.5461 | **0.9013** | 0.8289 |
| Balanced accuracy | 0.3960 | **0.8928** | 0.8086 |
| Macro F1 | 0.3706 | **0.9061** | 0.8179 |

The tabular-only Random Forest achieved the strongest performance across all three reported test metrics.

## 5. Per-Class Performance

### 5.1 Image-only CNN

| Class | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| A | 0.3590 | 0.2979 | 0.3256 | 47 |
| A+ | 0.6126 | 0.8500 | 0.7120 | 80 |
| B | 0.5000 | 0.0400 | 0.0741 | 25 |

The CNN performed best on class A+ but struggled to identify class B. It correctly classified only one of the 25 class B test records.

### 5.2 Tabular-only Random Forest

| Class | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| A | 0.8837 | 0.8085 | 0.8444 | 47 |
| A+ | 0.8837 | 0.9500 | 0.9157 | 80 |
| B | 1.0000 | 0.9200 | 0.9583 | 25 |

The Random Forest achieved strong performance across all three classes. It correctly classified 38 of 47 A cases, 76 of 80 A+ cases, and 23 of 25 B cases.

### 5.3 Multimodal Model

| Class | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| A | 0.8485 | 0.5957 | 0.7000 | 47 |
| A+ | 0.8085 | 0.9500 | 0.8736 | 80 |
| B | 0.8800 | 0.8800 | 0.8800 | 25 |

The multimodal model performed well on A+ and B but had difficulty distinguishing A from A+, with 16 of the 47 actual A cases classified as A+.

## 6. Confusion Matrices

Rows represent actual classes and columns represent predicted classes. The class order is A, A+, B.

### Image-only CNN

| Actual \ Predicted | A | A+ | B |
|---|---:|---:|---:|
| A | 14 | 32 | 1 |
| A+ | 12 | 68 | 0 |
| B | 13 | 11 | 1 |

### Tabular-only Random Forest

| Actual \ Predicted | A | A+ | B |
|---|---:|---:|---:|
| A | 38 | 9 | 0 |
| A+ | 4 | 76 | 0 |
| B | 1 | 1 | 23 |

### Multimodal Model

| Actual \ Predicted | A | A+ | B |
|---|---:|---:|---:|
| A | 28 | 16 | 3 |
| A+ | 4 | 76 | 0 |
| B | 1 | 2 | 22 |

## 7. Discussion

The tabular-only Random Forest outperformed the image-only CNN and the multimodal fusion model on the shared test set. Its macro F1 of 0.9061 was higher than the multimodal model's 0.8179 and the image-only CNN's 0.3706.

The image-only CNN showed a strong tendency to predict A+, indicating difficulty learning discriminative image representations for the three tissue grades under the current training configuration.

Although the multimodal model substantially outperformed the image-only CNN, combining image and tabular representations did not improve upon the tabular-only baseline. Potential explanations include limited training data, the simplicity of the image encoder, optimization challenges, or the strong predictive information already present in the tabular features.

These results do not establish that images provide no additional information. Further experiments are required to determine whether a better image representation can improve grading beyond the tabular baseline.

## 8. Limitations and Next Steps

Before drawing final conclusions, the following steps are required:

1. **Feature-leakage audit:** Verify that none of the selected input features directly encodes the target grade or a post-decision outcome.
2. **Reproducibility:** Preserve the shared split, preprocessing configuration, random seeds, and model settings.
3. **Image representation:** Investigate pretrained image encoders or alternative training strategies.
4. **Incremental value of images:** Compare tabular-only and multimodal models using a controlled experimental design.
5. **Statistical uncertainty:** Report confidence intervals and, where appropriate, paired comparisons of model predictions.
6. **Independent validation:** Evaluate the final approach on an independent dataset if one becomes available.

The test set has already been used for model evaluation, so its results should not be used for further hyperparameter tuning. The present findings should be treated as baseline results pending leakage checks and further validation.

## 9. Conclusion

Under the current experimental setup, the tabular-only Random Forest achieved the best observed performance, with a test accuracy of 90.13%, balanced accuracy of 89.28%, and macro F1 of 0.9061. The multimodal model achieved a macro F1 of 0.8179, while the image-only CNN achieved 0.3706.

The next priority is to verify the validity of the tabular predictors and investigate whether image-derived information provides measurable incremental value beyond the existing clinical and morphological features.
